<div align="center">

<a href="https://colab.research.google.com/github/utkukose/philosophy-of-ai-NB-lecture/blob/main/weeks/week-13/NB13_creativity.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Week 13: Creativity and Machines

**Philosophy of Artificial Intelligence (11118BLG003)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

*This course is updated in line with current developments in the field. Last update: September 2026.*

</div>

## How to use this notebook

This notebook is a self-contained version of the week: the lecture text comes first, followed by the hands-on lab. Run the cells in order with Shift+Enter. Exercise cells start with `None` placeholders and print feedback through `check(...)`. Reference solutions sit in collapsed cells: open them only after a genuine attempt. The interactive lab and the self-assessment are available at <https://utkukose.github.io/philosophy-of-ai-NB-lecture/weeks/week-13/lab.html>.

## Overview

Can a machine originate anything, or can it only do what it is told? This week examines Lovelace's objection and Turing's reply, Boden's analysis of creativity, criteria for attributing creativity to programs and search methods that reward novelty [1, 2, 3].

## Learning outcomes

By the end of the week, students are expected to state Lovelace's objection and Turing's reply, to distinguish combinational, exploratory and transformational creativity and P- from H-creativity, to apply the criteria of novelty, quality and typicality, and to explain how novelty search explores a conceptual space.

## Lovelace's objection

In her notes on Babbage's Analytical Engine, Ada Lovelace wrote that the engine had no pretensions to originate anything and could do only whatever people knew how to order it to perform [1]. Turing called this Lady Lovelace's objection and replied in two ways [4]. Machines often take people by surprise, because the consequences of their programs are not obvious to those who wrote them. More fundamentally, learning machines acquire behaviour that their programmers did not specify. The question for this week is whether surprise and learning amount to creativity.

## Three kinds of creativity

Boden analysed creativity as the production of ideas that are new, surprising and valuable, and distinguished three ways in which such ideas arise [2, 5]. Combinational creativity produces unfamiliar combinations of familiar ideas, as in poetic imagery or analogy. Exploratory creativity explores a structured conceptual space, such as a musical style, and finds possibilities that had not been realised. Transformational creativity changes the space itself, by altering one of its defining rules, so that ideas that were previously impossible become possible. Boden also distinguished psychological creativity, an idea new to the person who has it, from historical creativity, an idea new to anyone. She argued that computers can exhibit all three kinds, with transformational creativity the hardest to achieve and to evaluate.

L-systems, grammars introduced by Lindenmayer to model the development of organisms, give a concrete conceptual space [6, 7]. A rule such as replacing every segment with a branching pattern generates plant-like forms, as Figure 13.1 shows. Varying the angle or the number of iterations explores the space, while changing the grammar transforms it.

![Figure 13.1](https://raw.githubusercontent.com/utkukose/philosophy-of-ai-NB-lecture/main/weeks/week-13/figures/w13_fig1.png)

*Figure 13.1. Plant forms generated by L-systems. Changing parameters explores the space defined by a grammar, while changing the grammar transforms it [2, 6].*

## Evaluating creative systems

Ritchie proposed empirical criteria for attributing creativity to a program [3]. The central notions are novelty, the extent to which an output differs from existing examples, quality, the extent to which it is valued, and typicality, the extent to which it belongs to the intended genre. The criteria are relative to an inspiring set, the examples that guided the construction of the program. Wiggins formalised Boden's account in a creative systems framework that separates the conceptual space, the strategy for traversing it and the evaluation of results [8]. Colton and Wiggins described computational creativity as a field that studies systems that take on responsibilities in creative tasks, and noted that how a system's process is framed and perceived affects whether audiences regard it as creative [9].

## Searching for novelty

Lehman and Stanley showed that search guided only by the novelty of behaviour, with no objective, can outperform objective-driven search in deceptive problems, in which the path to a good solution leads away from it at first [10]. Novelty search keeps an archive of past behaviours and rewards candidates that are far from them, which makes it a mechanism for exploratory creativity. The lab lets students combine, explore and transform L-system plants and computes novelty against a growing gallery. The notebook compares novelty search with a search that maximises one objective and measures how much of the space each covers.

> **Pause and reflect.** In the lab, which of your plants would you call creative, and was the judgement yours or the program's? Where does the creativity lie?

# Hands-on lab

The notebook implements L-systems and a turtle interpreter, describes each plant by a few features, and compares novelty search with objective-driven search in the same space of rules [6, 10].

In [ ]:
# Setup: common libraries and the self-check helper used in the exercises.
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 2026

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = bool(np.allclose(np.asarray(value, dtype=float), np.asarray(expected, dtype=float), atol=tol))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] " + ("Correct." if ok else "Not yet. Revisit the explanation above and try again."))

## 1. L-systems and a turtle

In [ ]:
from matplotlib.collections import LineCollection

def lsystem(axiom, rules, n, max_len=60000):
    s = axiom
    for _ in range(n):
        s = "".join(rules.get(c, c) for c in s)
        if len(s) > max_len:
            break
    return s

def turtle(s, angle):
    x, y, a, stack, segs = 0.0, 0.0, 90.0, [], []
    for c in s:
        if c == "F":
            nx, ny = x + np.cos(np.radians(a)), y + np.sin(np.radians(a))
            segs.append(((x, y), (nx, ny))); x, y = nx, ny
        elif c == "+": a += angle
        elif c == "-": a -= angle
        elif c == "[": stack.append((x, y, a))
        elif c == "]": x, y, a = stack.pop()
    return segs

def draw(segs, ax, title=""):
    ax.add_collection(LineCollection(segs, colors="darkgreen", linewidths=0.7)); ax.autoscale(); ax.set_aspect("equal"); ax.axis("off"); ax.set_title(title, fontsize=8)

fig, axes = plt.subplots(1, 3, figsize=(9, 3.5))
for ax, (rule, ang) in zip(axes, [("F[+F]F[-F]F", 25.7), ("F[+F]F[-F][F]", 20), ("FF-[-F+F+F]+[+F-F-F]", 22.5)]):
    draw(turtle(lsystem("F", {"F": rule}, 4 if len(rule) < 15 else 3), ang), ax, rule)
plt.show()

## 2. A space of rules and a description of behaviour

A genome (a, b, c, angle) defines the rule F -> F^a [+F] F^b [-F] F^c. Each plant is described by its aspect ratio, its size and the horizontal spread of its tips.

In [ ]:
def plant(genome, n=3):
    a, b, c, ang = genome
    rule = "F" * a + "[+F]" + "F" * b + "[-F]" + "F" * c
    return turtle(lsystem("F", {"F": rule}, n), ang)

def features(segs):
    pts = np.array([p for s in segs for p in s])
    w, h = np.ptp(pts[:, 0]), np.ptp(pts[:, 1])
    return np.array([np.log1p(w / max(h, 1e-9)), np.log1p(len(segs)) / 5, np.abs(pts[:, 0]).mean() / max(h, 1e-9)])

def mutate(g, rng):
    a, b, c, ang = g
    a, b, c = [int(np.clip(v + rng.integers(-1, 2), 0, 3)) for v in (a, b, c)]
    return (a, b, c, float(np.clip(ang + rng.normal(0, 8), 5, 85)))

### Your turn, exercise 1

Write `novelty(f, archive, k=3)` that returns the mean Euclidean distance from the feature vector f to its k nearest neighbours in `archive`, a list of feature vectors.

In [ ]:
def novelty(f, archive, k=3):
    # Your code here
    return None

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _novelty_reference(f, archive, k=3):
    d = np.sort([np.linalg.norm(f - g) for g in archive])
    return float(d[:k].mean())

In [ ]:
arch = [features(plant(g)) for g in [(1, 1, 1, 25.0), (2, 0, 1, 20.0), (0, 2, 2, 40.0), (1, 2, 0, 60.0)]]
f_test = features(plant((3, 0, 0, 70.0)))
check("Exercise 1", novelty(f_test, arch), _novelty_reference(f_test, arch))

## 3. Novelty search versus objective search

Both searches mutate a population of genomes for 25 generations. Objective search keeps the tallest plants; novelty search keeps the most novel ones relative to an archive. Coverage counts the distinct cells of a coarse grid over the first two features that the search has visited.

In [ ]:
def search(mode, generations=25, pop=12, seed=SEED):
    rng = np.random.default_rng(seed)
    population = [(1, 1, 1, 25.0)] * pop
    archive, visited = [], set()
    for _ in range(generations):
        kids = [mutate(g, rng) for g in population for _ in range(2)]
        feats = [features(plant(g)) for g in kids]
        for f in feats:
            visited.add((int(f[0] * 4), int(f[1] * 4)))
        if mode == "objective":
            scores = [np.ptp(np.array([p for s in plant(g) for p in s])[:, 1]) for g in kids]
        else:
            ref = archive + feats
            scores = [_novelty_reference(f, ref, 4) for f in feats]
        order = np.argsort(scores)[::-1][:pop]
        population = [kids[i] for i in order]
        archive += [feats[i] for i in order[:2]]
    return len(visited), population

for mode in ["objective", "novelty"]:
    print(mode, "search visited", search(mode)[0], "cells")

### Your turn, exercise 2

Run novelty search with seeds 0 to 4 and compute the mean number of cells visited. Compare with objective search and explain in two sentences what this shows about exploratory creativity.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _coverage_reference():
    return float(np.mean([search("novelty", seed=s)[0] for s in range(5)]))

In [ ]:
mean_novelty_coverage = None  # your computation
check("Exercise 2", mean_novelty_coverage, _coverage_reference())
print("objective search, same seeds:", np.mean([search("objective", seed=s)[0] for s in range(5)]))

## Self-assessment and reflection

Take the self-assessment in the interactive lab: <https://utkukose.github.io/philosophy-of-ai-NB-lecture/weeks/week-13/lab.html> (tab: Check yourself). Then answer the reflection prompts:

1. Which of your lab plants would you call H-creative, P-creative or not creative at all? Justify your judgements.
2. Does Turing's appeal to surprise answer Lovelace's objection? Consider who is surprised and why.
3. Identify a result in your field that you would call transformational. Could a present AI system have produced it?

## Weekly task and submission

Write about 500 words that assess whether one generative AI system is creative in each of Boden's three senses, using Ritchie's criteria. Support the assessment with an example from the lab or the notebook, and attach the notebook with both exercises completed.

The weekly task supports self-learning and builds a personal portfolio. When the course is followed with the instructor during an active semester, the task can be sent together with the exported learning log to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation.

## Research and report assignment (optional)

**Creativity of generative systems.** Apply Boden's three kinds of creativity and Ritchie's criteria to one current generative system and argue whether the system, its users or its developers are the creative agents [2, 3, 9].

This research assignment is optional and supports self-learning. When the related weeks are followed within the course during an active semester, the report can be sent to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation. Unless the assignment states otherwise, a report has 1500 to 2500 words, follows the structure of an academic paper, cites at least six scholarly or official sources in square brackets and ends with a reference list.

## References

[1] Lovelace, A. A. (1843). Notes by the translator, in L. F. Menabrea, Sketch of the Analytical Engine invented by Charles Babbage. *Scientific Memoirs*, 3, 666-731.

[2] Boden, M. A. (2004). *The Creative Mind: Myths and Mechanisms* (2nd ed.). Routledge.

[3] Ritchie, G. (2007). Some empirical criteria for attributing creativity to a computer program. *Minds and Machines*, 17(1), 67-99. <https://doi.org/10.1007/s11023-007-9066-2>

[4] Turing, A. M. (1950). Computing machinery and intelligence. *Mind*, 59(236), 433-460. <https://doi.org/10.1093/mind/LIX.236.433>

[5] Boden, M. A. (1998). Creativity and artificial intelligence. *Artificial Intelligence*, 103(1-2), 347-356. <https://doi.org/10.1016/S0004-3702(98)00055-1>

[6] Lindenmayer, A. (1968). Mathematical models for cellular interactions in development I. Filaments with one-sided inputs. *Journal of Theoretical Biology*, 18(3), 280-299. <https://doi.org/10.1016/0022-5193(68)90079-9>

[7] Prusinkiewicz, P., & Lindenmayer, A. (1990). *The Algorithmic Beauty of Plants*. Springer.

[8] Wiggins, G. A. (2006). A preliminary framework for description, analysis and comparison of creative systems. *Knowledge-Based Systems*, 19(7), 449-458. <https://doi.org/10.1016/j.knosys.2006.04.009>

[9] Colton, S., & Wiggins, G. A. (2012). Computational creativity: The final frontier?. In *Proceedings of the 20th European Conference on Artificial Intelligence (ECAI 2012)* (pp. 21-26). IOS Press.

[10] Lehman, J., & Stanley, K. O. (2011). Abandoning objectives: Evolution through the search for novelty alone. *Evolutionary Computation*, 19(2), 189-223. <https://doi.org/10.1162/EVCO_a_00025>